In [ ]:
# idées LLMisées, pas forcéement pertinent, à voir

In [ ]:
import re
import pandas as pd
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
# TODO: aviser si vire id_orateur et utiliser id_acteur partout
df = pd.read_csv(
    "../data/interim/data_cleaning.csv", low_memory=False, dtype={"ID_orateur": str}
)
df.shape

(683680, 56)

In [ ]:
def nettoyer_texte(texte):
    if not isinstance(texte, str):
        return texte
    # Supprimer les balises HTML/XML
    texte = re.sub(r"<[^>]+>", "", texte)
    # Supprimer contenu entre parenthèses
    texte = re.sub(r"\([^)]*\)", "", texte)
    # Supprimer les espaces multiples
    texte = re.sub(r"\s+", " ", texte).strip()
    # uniformise pour avoir les bons apostrophes (nécessaire pour regex)
    texte = texte.replace("’", "'")
    return texte


df["Texte_clean"] = df["texte"].apply(nettoyer_texte)

In [ ]:
df = df.dropna(subset=["Texte_clean"])

In [ ]:
# préparer les pays à exclure
with open("../data/raw/liste_pays_republique_stable.txt", "r", encoding="utf-8") as f:
    liste_pays = [line.strip() for line in f]

# créer un pattern regex pour les pays
# ici pas besoin d'avoir un groupe de capture par pays mais juste global ok
pattern_pays = r"(\b(?:" + r"|".join(re.escape(p) for p in liste_pays) + r")\b)"


# TODO: check avec matthias si ok
# com matthias : Regex pays insuffisante car doit aussi comprendre la forme adjectivable des pays et république en minuscule
# > ajout d'une version stable avec forme adjectivable
# > et gestion de la casse (maj/min) dans la fonction par la regex (cf re.I)

# Pour matthias : j'ai pour l'instant gardé les \b (word boundary)
# cf. je pense que ton soucis avec République d'Arménie était à cause de l'apostrophe

# Regex du champ lexical République (simplifié ici)

pattern_lexical = re.compile(
    r"républi",  # même au milieu des mots
    re.I,
)

# Regex des expressions à exclure

# Expressions à exclure - casse exacte
pattern_excl_case_sensitive = re.compile(
    r"\b[LlDd]es Républicains\b"  # garde la casse pour identifier le parti (et pas un adjectif)
)  # voir pour élu Républicain ? doute

# Expressions à exclure - ignorer la casse
pattern_excl_case_insensitive = re.compile(
    # partis et groupes politiques
    r"|(\bgauche démocrate et républicaine)"
    r"|(\brépublique en marche\b)"
    r"|(\bsocialiste, écologiste et républicain\b)"
    # fonctions et institutions
    r"|(\bprésident[s]? de la République\b)"
    r"|(\bprésidence[s]? de la République\b)"
    r"|(\bprocureur[s]? de la République\b)"
    r"|(\bcour[s]? de justice de la République\b)"
    r"|(\bcour[s]? de sûreté de la République\b)"
    r"|(\badministration générale de la République\b)"
    r"|(\bGouvernement de la République française\b)"
    # pays
    r"|(\brépublique[s]? soviétique[s]?\b)"  # pas un pays mais des expressions
    r"|(" + pattern_pays + ")",  # ajout des exclusions de pays si existe
    re.I,
)


def contains_lexical_outside_excl(text):
    # Trouver les positions des expressions exclues
    excl_positions = []

    # Ajouter les exclusions sensibles à la casse
    excl_positions.extend(
        [m.span() for m in pattern_excl_case_sensitive.finditer(text)]
    )

    # Ajouter les exclusions insensibles à la casse
    excl_positions.extend(
        [m.span() for m in pattern_excl_case_insensitive.finditer(text)]
    )

    # Fonction pour vérifier si une position est dans une zone exclue
    def in_excl(pos):
        for start, end in excl_positions:
            if start <= pos < end:
                return True
        return False

    # Chercher toutes les occurences du champ lexical
    for match in pattern_lexical.finditer(text):
        start_pos = match.start()
        if not in_excl(start_pos):
            return True
    return False

In [ ]:
#################################################
# 2️⃣ Extraction des contextes non exclus
#################################################


def extract_contexts(series, keyword="républi", window=5):
    pattern = re.compile(rf"{keyword}", re.IGNORECASE)
    contexts = []

    for text in series.dropna():
        excl_positions = []

        excl_positions.extend(
            [m.span() for m in pattern_excl_case_sensitive.finditer(text)]
        )

        excl_positions.extend(
            [m.span() for m in pattern_excl_case_insensitive.finditer(text)]
        )

        def in_excl(pos):
            for start, end in excl_positions:
                if start <= pos < end:
                    return True
            return False

        tokens = re.findall(r"\b[\w-]+\b", text)

        char_index = 0
        token_positions = []

        for tok in tokens:
            pos = text.find(tok, char_index)
            token_positions.append((tok, pos))
            char_index = pos + len(tok)

        for i, (token, pos) in enumerate(token_positions):
            if pattern.search(token):
                if in_excl(pos):
                    continue

                start = max(0, i - window)
                end = min(len(tokens), i + window + 1)

                context = " ".join(tokens[start:end])

                contexts.append(context)

    return contexts


#################################################
# 3️⃣ Clustering des contextes
#################################################


def cluster_contexts(
    contexts, n_clusters=10, model_name="paraphrase-multilingual-MiniLM-L12-v2"
):
    model = SentenceTransformer(model_name)

    embeddings = model.encode(contexts, show_progress_bar=True)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42)

    labels = kmeans.fit_predict(embeddings)

    return labels


#################################################
# 4️⃣ Affichage des clusters
#################################################


def display_clusters(contexts, labels, top_n=10):
    df = pd.DataFrame({"context": contexts, "cluster": labels})

    for cluster_id in sorted(df.cluster.unique()):
        cluster_texts = df[df.cluster == cluster_id]["context"]

        print("\n=====================================")
        print(f"CLUSTER {cluster_id} ({len(cluster_texts)} occurrences)")
        print("=====================================\n")

        for c in cluster_texts.head(top_n):
            print("-", c)


#################################################
# 5️⃣ Keywords par cluster
#################################################


def show_cluster_keywords(contexts, labels, top_k=10):
    df = pd.DataFrame({"context": contexts, "cluster": labels})

    for cluster_id in sorted(df.cluster.unique()):
        texts = df[df.cluster == cluster_id]["context"]

        vectorizer = CountVectorizer(ngram_range=(1, 3), stop_words=None)

        X = vectorizer.fit_transform(texts)

        counts = X.sum(axis=0).A1
        vocab = vectorizer.get_feature_names_out()

        top = sorted(zip(vocab, counts), key=lambda x: -x[1])[:top_k]

        print(f"\nKeywords cluster {cluster_id}\n")

        for w, c in top:
            print(w, c)


#################################################
# 6️⃣ Génération de motifs regex candidats
#################################################


def build_regex_from_clusters(contexts, labels, keyword="république", min_freq=5):
    df = pd.DataFrame({"context": contexts, "cluster": labels})

    regex_candidates = []

    for cluster_id in sorted(df.cluster.unique()):
        texts = df[df.cluster == cluster_id]["context"]

        vectorizer = CountVectorizer(ngram_range=(2, 3), stop_words=None)

        X = vectorizer.fit_transform(texts)

        counts = X.sum(axis=0).A1
        vocab = vectorizer.get_feature_names_out()

        pairs = [
            (vocab[i], counts[i])
            for i in range(len(vocab))
            if counts[i] >= min_freq and keyword in vocab[i]
        ]

        regex_candidates.extend(pairs)

    regex_candidates = sorted(regex_candidates, key=lambda x: -x[1])

    return regex_candidates


#################################################
# 7️⃣ Pipeline complet
#################################################


def cluster_pipeline(series, keyword="républi", window=5, n_clusters=8):
    print("Extraction des contextes...")

    contexts = extract_contexts(series, keyword, window)

    print("Nombre de contextes trouvés:", len(contexts))

    print("\nEmbedding + clustering...")

    labels = cluster_contexts(contexts, n_clusters)

    print("\nClusters détectés :")

    display_clusters(contexts, labels)

    print("\nKeywords par cluster :")

    show_cluster_keywords(contexts, labels)

    print("\nMotifs candidats pour exclusions regex :")

    patterns = build_regex_from_clusters(contexts, labels)

    for p in patterns[:30]:
        print(p)

    return contexts, labels, patterns


#################################################
# 8️⃣ Exemple d'utilisation
#################################################

# df = pd.read_csv("ton_corpus.csv")

# contexts, labels, patterns = cluster_pipeline(
#     df["texte"],
#     keyword="républi",
#     window=6,
#     n_clusters=8
# )

In [ ]:
contexts, labels, patterns = cluster_pipeline(
    df["Texte_clean"], keyword="républi", window=6, n_clusters=8
)

Extraction des contextes...
Nombre de contextes trouvés: 19680

Embedding + clustering...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/615 [00:00<?, ?it/s]


Clusters détectés :

CLUSTER 0 (4640 occurrences)

- c est la grandeur de la République et de sa politique sociale que
- dans ce haut lieu de la République et de la démocratie
- font l obligation d assurer une République solide c est-à-dire un État de
- de droit le fondement de notre République est la liberté La liberté ne
- s il est vital que la République investisse durablement dans la lutte contre
- un des premiers devoirs de la République est de garantir la sécurité de
- vous êtes un ministre de la République qui répond devant la représentation nationale
- ce combat est celui de la République Avec quels moyens le menons-nous dans
- cause de lieux qui symbolisent notre République et notre démocratie à dire son
- Parce qu ensemble nous sommes la République

CLUSTER 1 (1178 occurrences)

- pas le monopole ni des valeurs républicaines ni du patriotisme ni non plus
- résolution il est clair que le Républicain n est pas là dans son
- même où notre société laïque et républicaine subit un

In [ ]:
show_cluster_keywords(contexts, labels)



Keywords cluster 0

la 5169
république 4785
de 4429
la république 3452
de la 2574
de la république 2197
et 1571
est 1072
les 1070
le 1066

Keywords cluster 1

de 571
la 407
républicain 405
républicaine 376
est 332
et 332
pas 265
républicains 261
les 260
le 235

Keywords cluster 2

républicain 632
contrat 491
engagement 430
engagement républicain 413
contrat engagement 407
contrat engagement républicain 394
le 392
de 317
le contrat 210
le contrat engagement 186

Keywords cluster 3

république 3575
la 3528
de 2557
la république 2534
de la 1413
de la république 1214
est 941
pas 923
et 790
les 707

Keywords cluster 4

la 5571
de 5148
république 4817
la république 3498
de la 3183
de la république 2543
le 1274
les 1250
des 974
et 944

Keywords cluster 5

de 1053
la 739
et 686
républicaine 656
républicain 521
les 431
républicains 419
le 411
nous 406
est 395

Keywords cluster 6

de 1345
la 843
républicaine 752
républicain 690
et 584
le 509
les 449
des 387
est 329
que 294

Keywords cluster 7

